## Problem
- Bank data with customer details
- Output = how many customers exited or stayed (churn rate)

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [62]:
# Load the dataset

data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
# preprocess data - drop irrelevant features
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
# Encode categorical variables

# for gender a 0/1 works because there are only 2 values
# but for other categories there are many different values, and higher values will get higher weights in model
# so we cannot directly use label encoder for them
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
# onehot encode Geography
onehot_encoder_geography = OneHotEncoder(sparse_output=False)
geo_encoder = onehot_encoder_geography.fit_transform(data[['Geography']])
print(geo_encoder)
print(onehot_encoder_geography.get_feature_names_out())

[[1. 0. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 ...
 [1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]]
['Geography_France' 'Geography_Germany' 'Geography_Spain']


In [14]:
geo_encoded_df = pd.DataFrame(geo_encoder, columns=onehot_encoder_geography.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [ ]:
# combining one hot encoded columns with the original data and removing original geography column
data = pd.concat((data.drop('Geography', axis=1), geo_encoded_df), axis=1)
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [ ]:
# save the encoders and scaler as pickle file
# wb = write, binary mode
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geography.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geography, file)

In [18]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [19]:
# divide the dataset into independent and dependent features (output, input)
X = data.drop('Exited', axis=1)
y = data['Exited']

# test train split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scale these features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [21]:
X_train[0]

array([ 0.35649971,  0.91324755, -0.6557859 ,  0.34567966, -1.21847056,
        0.80843615,  0.64920267,  0.97481699,  1.36766974,  1.00150113,
       -0.57946723, -0.57638802])

In [22]:
# save scaler model
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

## ANN Implementation

- in tensorflow, ANN is called a Sequential Model
- in keras, hidden neurons are called Dense
    - Dense 64 means current hidden layer will have 64 nodes
- activation function: sigmoid, tanh, relu, leaky relu
- optimizer is used for back propogation for updating weights
- early stopping
    - used to stop model if it has trained well (not much variation anymore) before all epochs complete
- loss function
- metrics
    - classification -> accuracy
    - regression -> mse, mae
- training logs stored in a folder using TensorBoard -> visualization graphs

In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [25]:
# build ann
# input shape for hidden layer 1 (tuple of dimension) = (X_train.shape[1], )
# subsequent hidden layers don't need input shape, because it takes the input shape from previous layer itself

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)), # Hidden layer 1
    Dense(32, activation='relu'), # Hidden layer 2
    Dense(1, activation='sigmoid') # Output layer
])

/Users/y0a02i0/Documents/Udemy/Complete Gen AI Course with Langchain and HuggingFace/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [26]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# compile the model
# two ways to specify loss and optimizer:
# 1
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
# loss = tf.keras.losses.BinaryCrossentropy()

# 2
# model.compile(optimizer="adam", loss="binary_crossentropy", metrics=['accuracy'])

In [27]:
# compiling model using combination
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=['accuracy'])

In [ ]:
# setup tensorboard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tf_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [31]:
# setup early stopping
# we want to monitor the validation loss, wait for atleast 5 epochs to make sure not much variation before stopping, and restore the best weights encountered before stopping
early_stopping_callback = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

In [32]:
# train model
history = model.fit(
    X_train, y_train, validation_data=(X_test, y_test), epochs=100,
    callbacks=[tf_callback, early_stopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 680us/step - accuracy: 0.8640 - loss: 0.3332 - val_accuracy: 0.8630 - val_loss: 0.3449
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 595us/step - accuracy: 0.8637 - loss: 0.3347 - val_accuracy: 0.8645 - val_loss: 0.3355
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 570us/step - accuracy: 0.8636 - loss: 0.3316 - val_accuracy: 0.8625 - val_loss: 0.3519
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 597us/step - accuracy: 0.8649 - loss: 0.3290 - val_accuracy: 0.8595 - val_loss: 0.3431
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 598us/step - accuracy: 0.8666 - loss: 0.3252 - val_accuracy: 0.8605 - val_loss: 0.3493
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 582us/step - accuracy: 0.8652 - loss: 0.3267 - val_accuracy: 0.8635 - val_loss: 0.3383
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 594us/step - accuracy: 0.8648 - loss: 0.3249 - val_accuracy: 0.8540 - val_loss: 0.3499
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 576us/step - accuracy: 0.8669 - loss: 0

In [33]:
# save model
model.save('model.h5')

In [1]:
# load tensorboard extension
%load_ext tensorboard

In [ ]:
# tensorboard not supported for python 3.13
# hence using google colab to load tensorboard by importing logs folder there - https://colab.research.google.com/drive/1hnSjjy-NlWYIOZF10XvSl3b0u49gZ7zq#scrollTo=Q446AinKtD9D
%tensorboard --logdir logs/fit20260320-102601/

ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
Traceback (most recent call last):
  File "/Users/y0a02i0/Documents/Udemy/Complete Gen AI Course with Langchain and HuggingFace/.venv/bin/tensorboard", line 5, in <module>
    from tensorboard.main import run_main
  File "/Users/y0a02i0/Documents/Udemy/Complete Gen AI Course with Langchain and HuggingFace/.venv/lib/python3.13/site-packages/tensorboard/main.py", line 27, in <module>
    from tensorboard import default
  File "/Users/y0a02i0/Documents/Udemy/Complete Gen AI Course with Langchain and HuggingFace/.venv/lib/python3.13/site-packages/tensorboard/default.py", line 30, in <module>
    import pkg_resources
ModuleNotFoundError: No module named 'pkg_resources'

## Prediction

In [47]:
# load the pickle file
model = load_model('model.h5')

# open in read binary mode
with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geography.pkl', 'rb') as file:
    onehot_encoder_geography = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [48]:
# Example input data for prediction
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

# geo encode geography column
geo_encoded = onehot_encoder_geography.transform([[input_data['Geography']]])
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geography.get_feature_names_out())
geo_encoded_df

/Users/y0a02i0/Documents/Udemy/Complete Gen AI Course with Langchain and HuggingFace/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [49]:
# convert dictionary to df
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [50]:
input_df = pd.concat([input_df.drop('Geography', axis=1), geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [ ]:
# encode categorical variables
# input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

# above line should work but its not working, will have to run entire code again to fit again and save model again with complete training
input_df['Gender'] = 1
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [58]:
# scaling the data
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [61]:
# prediction
prediction = model.predict(input_scaled)
prediction[0][0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


np.float32(0.018145239)

## Streamlit App

In [ ]:
# in app.py